In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os, joblib
import numpy as np
import pandas as pd
import polars as pl

import kaggle_evaluation.default_inference_server

from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from sklearn.linear_model import RidgeCV

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.model_selection import train_test_split

In [2]:
# Read data - DON'T drop nulls immediately
train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

def preprocessing(data, typ, create_interactions=True, add_lag_features=True):
    """
    Enhanced preprocessing with advanced feature engineering
    
    Parameters:
    -----------
    data : DataFrame
        Input market data
    typ : str
        One of 'train', 'test', or 'inference'
    create_interactions : bool
        Whether to create interaction features
    add_lag_features : bool
        Whether to add lagged features
    """
    
    # Define feature groups
    feature_groups = {
        'market': [f'M{i}' for i in range(1, 19)],
        'economic': [f'E{i}' for i in range(1, 21)],
        'interest': [f'I{i}' for i in range(1, 10)],
        'price': ['P8', 'P9', 'P10', 'P12', 'P13'],
        'volatility': [f'V{i}' for i in range(1, 14)],
        'sentiment': [f'S{i}' for i in range(1, 13)],
        'dummy': [f'D{i}' for i in range(1, 10)]
    }
    
    main_features = [f for group in feature_groups.values() for f in group]
    
    # Select base features and target
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    elif typ == "test":
        # Test set should not have target
        data = data[main_features].copy()
    else:  # inference
        data = data[main_features].copy()
    
    # ========== ADVANCED FEATURE ENGINEERING ==========
    
    # 1. Statistical Features by Group
    for group_name, features in feature_groups.items():
        available_features = [f for f in features if f in data.columns]
        if len(available_features) > 1:
            # Basic statistics
            data[f'{group_name}_mean'] = data[available_features].mean(axis=1)
            data[f'{group_name}_std'] = data[available_features].std(axis=1)
            data[f'{group_name}_max'] = data[available_features].max(axis=1)
            data[f'{group_name}_min'] = data[available_features].min(axis=1)
            data[f'{group_name}_range'] = data[f'{group_name}_max'] - data[f'{group_name}_min']
            
            # Advanced statistics
            data[f'{group_name}_median'] = data[available_features].median(axis=1)
            data[f'{group_name}_skew'] = data[available_features].skew(axis=1)
            data[f'{group_name}_kurt'] = data[available_features].kurtosis(axis=1)
            
            # Quartiles and IQR
            data[f'{group_name}_q25'] = data[available_features].quantile(0.25, axis=1)
            data[f'{group_name}_q75'] = data[available_features].quantile(0.75, axis=1)
            data[f'{group_name}_iqr'] = data[f'{group_name}_q75'] - data[f'{group_name}_q25']
            
            # Coefficient of variation (normalized volatility)
            data[f'{group_name}_cv'] = data[f'{group_name}_std'] / (data[f'{group_name}_mean'].abs() + 1e-8)
            
            # Count of positive/negative features
            data[f'{group_name}_pos_count'] = (data[available_features] > 0).sum(axis=1)
            data[f'{group_name}_neg_count'] = (data[available_features] < 0).sum(axis=1)
            data[f'{group_name}_pos_ratio'] = data[f'{group_name}_pos_count'] / len(available_features)
    
    # 2. Cross-Group Interactions (important combinations)
    if create_interactions:
        # Market-Volatility interaction (common in finance)
        if 'market_mean' in data.columns and 'volatility_mean' in data.columns:
            data['market_vol_interaction'] = data['market_mean'] * data['volatility_mean']
            data['market_vol_ratio'] = data['market_mean'] / (data['volatility_mean'].abs() + 1e-8)
        
        # Sentiment-Volatility interaction
        if 'sentiment_mean' in data.columns and 'volatility_mean' in data.columns:
            data['sentiment_vol_interaction'] = data['sentiment_mean'] * data['volatility_mean']
            data['sentiment_vol_ratio'] = data['sentiment_mean'] / (data['volatility_std'].abs() + 1e-8)
        
        # Economic-Interest rate interaction
        if 'economic_mean' in data.columns and 'interest_mean' in data.columns:
            data['econ_interest_interaction'] = data['economic_mean'] * data['interest_mean']
            data['econ_interest_diff'] = data['economic_mean'] - data['interest_mean']
            data['econ_interest_ratio'] = data['economic_mean'] / (data['interest_mean'].abs() + 1e-8)
        
        # Price-Volatility (risk-adjusted returns indicator)
        if 'price_mean' in data.columns and 'volatility_mean' in data.columns:
            data['price_vol_ratio'] = data['price_mean'] / (data['volatility_mean'].abs() + 1e-8)
        
        # Market-Sentiment (market psychology)
        if 'market_mean' in data.columns and 'sentiment_mean' in data.columns:
            data['market_sentiment_alignment'] = data['market_mean'] * data['sentiment_mean']
            data['market_sentiment_divergence'] = data['market_mean'] - data['sentiment_mean']
        
        # Economic-Market strength
        if 'economic_mean' in data.columns and 'market_mean' in data.columns:
            data['econ_market_strength'] = data['economic_mean'] * data['market_mean']
        
        # Market momentum (if multiple market features available)
        market_cols = [c for c in data.columns if c.startswith('M') and c[1:].isdigit()]
        if len(market_cols) >= 2:
            data['market_momentum'] = data[market_cols[-1]] - data[market_cols[0]]
            data['market_momentum_pct'] = (data[market_cols[-1]] - data[market_cols[0]]) / (data[market_cols[0]].abs() + 1e-8)
    
    # 3. Volatility-based features
    volatility_cols = [c for c in data.columns if c.startswith('V') and len(c) <= 3]
    if len(volatility_cols) > 0:
        # Volatility change (difference between high and low vol measures)
        if 'V1' in data.columns and 'V13' in data.columns:
            data['vol_change'] = data['V13'] - data['V1']
            data['vol_change_pct'] = (data['V13'] - data['V1']) / (data['V1'].abs() + 1e-8)
        
        # Volatility trends
        if 'V1' in data.columns and 'V5' in data.columns and 'V10' in data.columns:
            data['vol_short_trend'] = data['V5'] - data['V1']
            data['vol_long_trend'] = data['V10'] - data['V1']
            data['vol_acceleration'] = data['vol_long_trend'] - data['vol_short_trend']
        
        # Coefficient of variation for volatility
        if 'volatility_mean' in data.columns and 'volatility_std' in data.columns:
            data['vol_cv'] = data['volatility_std'] / (data['volatility_mean'].abs() + 1e-8)
            
        # Volatility regime (high/low vol periods)
        if 'volatility_mean' in data.columns and 'volatility_median' in data.columns:
            data['vol_regime'] = (data['volatility_mean'] > data['volatility_median']).astype(int)
            data['vol_extremity'] = (data['volatility_mean'] - data['volatility_median']) / (data['volatility_std'] + 1e-8)
    
    # 4. Sentiment indicators
    sentiment_cols = [c for c in data.columns if c.startswith('S') and len(c) <= 3]
    if len(sentiment_cols) > 0:
        # Sentiment strength
        if 'sentiment_mean' in data.columns:
            data['sentiment_strength'] = data['sentiment_mean'].abs()
            data['sentiment_direction'] = np.sign(data['sentiment_mean'])
        
        # Sentiment dispersion (disagreement)
        if 'sentiment_std' in data.columns and 'sentiment_mean' in data.columns:
            data['sentiment_dispersion'] = data['sentiment_std'] / (data['sentiment_mean'].abs() + 1e-8)
            data['sentiment_consensus'] = 1 / (data['sentiment_std'] + 1e-8)
        
        # Sentiment extremes
        if 'sentiment_max' in data.columns and 'sentiment_min' in data.columns:
            data['sentiment_extreme_spread'] = data['sentiment_max'] - data['sentiment_min']
            data['sentiment_extreme_ratio'] = data['sentiment_max'] / (data['sentiment_min'].abs() + 1e-8)
        
        # Sentiment momentum
        if 'S1' in data.columns and 'S12' in data.columns:
            data['sentiment_momentum'] = data['S12'] - data['S1']
            
        # Sentiment alignment (positive vs negative count)
        if 'sentiment_pos_ratio' in data.columns:
            data['sentiment_polarization'] = np.abs(data['sentiment_pos_ratio'] - 0.5) * 2
    
    # 5. Risk indicators
    if 'interest_mean' in data.columns and 'volatility_mean' in data.columns:
        data['risk_indicator'] = data['volatility_mean'] / (data['interest_mean'].abs() + 1e-8)
        data['risk_premium'] = data['volatility_mean'] - data['interest_mean']
    
    # Sharpe-like ratio (return/volatility proxy)
    if 'market_mean' in data.columns and 'volatility_mean' in data.columns:
        data['sharpe_proxy'] = data['market_mean'] / (data['volatility_mean'].abs() + 1e-8)
    
    # Risk-adjusted momentum
    if 'market_momentum' in data.columns and 'volatility_mean' in data.columns:
        data['risk_adj_momentum'] = data['market_momentum'] / (data['volatility_mean'].abs() + 1e-8)
    
    # 6. Price momentum and ratios
    price_cols = [c for c in data.columns if c in ['P8', 'P9', 'P10', 'P12', 'P13']]
    if len(price_cols) >= 2:
        data['price_momentum'] = data[price_cols[-1]] - data[price_cols[0]]
        data['price_momentum_pct'] = (data[price_cols[-1]] - data[price_cols[0]]) / (data[price_cols[0]].abs() + 1e-8)
        
        if 'P8' in data.columns and 'P13' in data.columns:
            data['price_ratio'] = data['P13'] / (data['P8'].abs() + 1e-8)
        
        # Price trends
        if 'price_mean' in data.columns and 'price_median' in data.columns:
            data['price_skewness_indicator'] = data['price_mean'] - data['price_median']
        
        # Price volatility
        if 'price_std' in data.columns and 'price_mean' in data.columns:
            data['price_volatility'] = data['price_std'] / (data['price_mean'].abs() + 1e-8)
    
    # Price-Market relationship
    if 'price_mean' in data.columns and 'market_mean' in data.columns:
        data['price_market_correlation_proxy'] = data['price_mean'] * data['market_mean']
    
    # 7. Economic strength composite
    if 'economic_mean' in data.columns and 'interest_mean' in data.columns:
        data['economic_strength'] = data['economic_mean'] - data['interest_mean']
        data['economic_interest_balance'] = data['economic_mean'] / (data['interest_mean'].abs() + 1e-8)
    
    # Economic momentum
    economic_cols = [c for c in data.columns if c.startswith('E') and c[1:].isdigit()]
    if len(economic_cols) >= 2:
        data['economic_momentum'] = data[economic_cols[-1]] - data[economic_cols[0]]
    
    # Economic volatility (uncertainty)
    if 'economic_std' in data.columns:
        data['economic_uncertainty'] = data['economic_std']
        if 'economic_mean' in data.columns:
            data['economic_uncertainty_ratio'] = data['economic_std'] / (data['economic_mean'].abs() + 1e-8)
    
    # 8. Lagged features (if requested)
    # Create lag features for all types (train/test/inference) to ensure consistency
    if add_lag_features:
        feature_cols = [c for c in data.columns if c != 'target']
        # Add 1-period and 2-period lags for key aggregated features
        agg_features = [c for c in feature_cols if '_mean' in c or '_std' in c or 'momentum' in c]
        for col in agg_features[:15]:  # Top 15 most important aggregates
            data[f'{col}_lag1'] = data[col].shift(1)
            
        # Rolling statistics (3-period and 5-period windows)
        key_features = ['market_mean', 'volatility_mean', 'sentiment_mean', 'economic_mean']
        for col in key_features:
            if col in data.columns:
                data[f'{col}_roll3_mean'] = data[col].rolling(window=3, min_periods=1).mean()
                data[f'{col}_roll3_std'] = data[col].rolling(window=3, min_periods=1).std()
                data[f'{col}_roll5_mean'] = data[col].rolling(window=5, min_periods=1).mean()
                
                # Exponential moving averages
                data[f'{col}_ema3'] = data[col].ewm(span=3, adjust=False).mean()
                data[f'{col}_ema5'] = data[col].ewm(span=5, adjust=False).mean()
                
                # Rate of change
                data[f'{col}_roc3'] = (data[col] - data[col].shift(3)) / (data[col].shift(3).abs() + 1e-8)
    
    # 9. Composite indicators (combining multiple features)
    # Market regime indicator
    if 'market_mean' in data.columns and 'volatility_mean' in data.columns and 'sentiment_mean' in data.columns:
        data['market_regime_composite'] = (
            data['market_mean'] * 0.4 + 
            data['sentiment_mean'] * 0.3 - 
            data['volatility_mean'] * 0.3
        )
    
    # Risk-on/Risk-off indicator
    if 'volatility_mean' in data.columns and 'sentiment_mean' in data.columns and 'market_mean' in data.columns:
        data['risk_on_indicator'] = (
            data['market_mean'] + data['sentiment_mean'] - data['volatility_mean']
        )
    
    # Macro health score
    if 'economic_mean' in data.columns and 'interest_mean' in data.columns:
        data['macro_health_score'] = data['economic_mean'] - data['interest_mean'] * 0.5
        if 'sentiment_mean' in data.columns:
            data['macro_health_score'] += data['sentiment_mean'] * 0.3
    
    # ========== HANDLE MISSING VALUES ==========
    
    # Separate target column
    target_col = None
    if 'target' in data.columns:
        target_col = data['target'].copy()
        data = data.drop('target', axis=1)
    
    # Fill missing values
    # For numeric columns, use forward fill then backward fill, then mean
    for col in data.columns:
        if data[col].dtype in ['float64', 'float32', 'int64', 'int32']:
            # Forward fill for time series nature
            data[col] = data[col].fillna(method='ffill')
            # Backward fill for remaining
            data[col] = data[col].fillna(method='bfill')
            # Mean for any remaining
            data[col] = data[col].fillna(data[col].mean())
            # If still NaN (all values were NaN), fill with 0
            data[col] = data[col].fillna(0)
    
    # Add target back
    if target_col is not None:
        data['target'] = target_col
        # Drop rows with null targets
        data = data.dropna(subset=['target'])
    
    return data


def get_feature_importance_groups(feature_names):
    """
    Helper function to group features for interpretability
    """
    groups = {
        'base_market': [],
        'base_economic': [],
        'base_interest': [],
        'base_price': [],
        'base_volatility': [],
        'base_sentiment': [],
        'base_dummy': [],
        'aggregated': [],
        'interactions': [],
        'momentum': [],
        'lagged': []
    }
    
    for name in feature_names:
        if name.startswith('M') and name[1:].isdigit():
            groups['base_market'].append(name)
        elif name.startswith('E') and name[1:].isdigit():
            groups['base_economic'].append(name)
        elif name.startswith('I') and name[1:].isdigit():
            groups['base_interest'].append(name)
        elif name.startswith('P') and name[1:].isdigit():
            groups['base_price'].append(name)
        elif name.startswith('V') and name[1:].isdigit():
            groups['base_volatility'].append(name)
        elif name.startswith('S') and name[1:].isdigit():
            groups['base_sentiment'].append(name)
        elif name.startswith('D') and name[1:].isdigit():
            groups['base_dummy'].append(name)
        elif '_mean' in name or '_std' in name or '_max' in name or '_min' in name or '_range' in name:
            groups['aggregated'].append(name)
        elif 'interaction' in name or 'ratio' in name:
            groups['interactions'].append(name)
        elif 'momentum' in name or 'change' in name:
            groups['momentum'].append(name)
        elif '_lag' in name:
            groups['lagged'].append(name)
    
    return groups
    
train = preprocessing(train, 'train', add_lag_features=True)

train_split, val_split = train_test_split(
    train, test_size=0.1, random_state=42
)

train_x = train_split.drop(columns=["target"])
test_x = val_split.drop(columns=["target"])
train_y = train_split['target']
test_y = val_split['target']

In [3]:
train

,M1,M2,M3,M4,M5,M6,M7,M8,M9,M10,...,economic_mean_roll3_mean,economic_mean_roll3_std,economic_mean_roll5_mean,economic_mean_ema3,economic_mean_ema5,economic_mean_roc3,market_regime_composite,risk_on_indicator,macro_health_score,target
0,1.639695,1.470488,-0.633441,0.308196,0.759627,0.580828,1.193217,0.000661,1.147322,0.546719,...,0.368239,0.011877,0.368239,0.368239,0.368239,-0.016155,-0.001874,-0.059988,0.434470,-0.002421
1,1.639695,1.470488,-0.633441,0.308196,0.759627,0.580828,1.193217,0.000661,1.147322,0.546719,...,0.368239,0.011877,0.368239,0.368239,0.368239,-0.016155,-0.001874,-0.059988,0.434470,-0.008495
2,1.639695,1.470488,-0.633441,0.308196,0.759627,0.580828,1.193217,0.000661,1.147322,0.546719,...,0.368239,0.011877,0.368239,0.368239,0.368239,-0.016155,-0.001874,-0.059988,0.434470,-0.009624
3,1.639695,1.470488,-0.633441,0.308196,0.759627,0.580828,1.193217,0.000661,1.147322,0.546719,...,0.368239,0.011877,0.368239,0.368239,0.368239,-0.016155,-0.001874,-0.059988,0.434470,0.004662
4,1.639695,1.470488,-0.633441,0.308196,0.759627,0.580828,1.193217,0.000661,1.147322,0.546719,...,0.368239,0.011877,0.368239,0.368239,0.368239,-0.016155,-0.001874,-0.059988,0.434470,-0.011686
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,-1.190313,-0.632252,1.138825,0.152280,1.359579,-0.187969,-0.571545,0.731812,0.331379,-0.262267,...,0.365007,0.013442,0.357562,0.367127,0.359507,0.062144,0.012496,0.024332,0.212496,0.002457
8986,-1.218842,-0.418564,2.620436,0.097712,0.909058,-0.155919,-0.571308,0.772817,0.341697,-0.288451,...,0.362860,0.015532,0.360918,0.359221,0.356777,-0.018007,0.099810,0.292402,0.243448,0.002312
8987,-1.234393,0.501880,0.950040,0.061329,1.428299,-0.277512,-0.571071,0.854167,0.355097,-0.292948,...,0.360524,0.017333,0.359215,0.354479,0.354430,-0.019644,0.083654,0.245458,0.225421,0.002891
8988,-1.297210,-0.222224,0.948951,-0.006532,1.518843,-0.171653,-0.570834,0.773810,0.370644,-0.302140,...,0.354366,0.006697,0.360072,0.358262,0.356968,-0.048548,-0.000547,-0.020026,0.171562,0.008310


In [4]:
catboost_params = {'iterations' : 5000,
                   'learning_rate': 0.009, 
                   'depth': 5, 
                   'l2_leaf_reg': 5.5,
                   'min_child_samples' : 102,
                   'od_wait' : 50,
                   'random_state' : 42,
                   'eval_metric': 'MAE', 
                   'od_type' : 'Iter',
                   'bootstrap_type': 'Bayesian', 
                   'grow_policy' : 'Depthwise',
                   'logging_level' : 'Silent'}

LGBM_R_parm = {'boosting_type': 'gbdt', 
               'colsample_bytree': 0.9484106149593443, 
               'learning_rate': 0.1988123373955639, 
               'max_bin': 77, 
               'max_depth': 10, 
               'metric': 'mape', 
               'min_child_samples': 81, 
               'min_data_in_leaf': 21, 
               'n_estimators': 5029, 
               'num_leaves': 42, 
               'objective': 'regression_l1', 
               'reg_alpha': 0.6355835028602363, 
               'reg_lambda': 3.109823217156622, 
               'subsample': 0.7300733288106989, 
               'verbosity': -1}

XGB_R_parm = {'colsample_bytree': 0,
              'enable_categorical': True,
              'gamma': 5,
              'learning_rate': 0.18348831817680378,
              'max_depth': 15,
              'min_child_weight': 1,
              'n_estimators': 5000,
              'objective': 'reg:squarederror',
              'reg_alpha': 94,
              'reg_lambda': 0.41318910368801975,
              'subsample': 0.5444965693077323}

R_Forest_parm = {'criterion': 'absolute_error', 
                 'max_depth': 6, 'max_features': 'sqrt', 
                 'max_leaf_nodes': 42, 
                 'min_samples_leaf': 6, 
                 'min_samples_split': 13, 
                 'min_weight_fraction_leaf': 0.0023062425041415757, 
                 'n_estimators': 108, 
                 'random_state': 42}

Extra_parm = {'criterion': 'absolute_error', 
              'max_depth': 6, 'max_features': 'sqrt', 
              'max_leaf_nodes': 42, 
              'min_samples_leaf': 6, 
              'min_samples_split': 13, 
              'min_weight_fraction_leaf': 0.0023062425041415757, 
              'n_estimators': 108, 
              'random_state': 42}

GB_params = {'learning_rate' : 0.1, 
             'min_samples_split' : 500,
             'min_samples_leaf' : 50,
             'max_depth' : 8,
             'max_features' : 'sqrt',
             'subsample' : 0.8,
             'random_state' : 10}

print(f'CatBoostRegressor TRAINING...')
catboost = CatBoostRegressor(**catboost_params)
cat_features = list(train_x.select_dtypes(include=['object', 'category']).columns)
train_pool = Pool(train_x, train_y, cat_features=cat_features)
val_pool = Pool(test_x, test_y, cat_features=cat_features)
#catboost.fit(train_pool, eval_set=(val_pool), verbose=100, early_stopping_rounds=100)
joblib.dump(catboost, 'catboost.pkl')

print(f'LGBMRegressor TRAINING...')
LGBM_R = LGBMRegressor(**LGBM_R_parm)
LGBM_R.fit(train_x, train_y, eval_set=[(test_x, test_y)])
joblib.dump(LGBM_R, 'LGBM_R.pkl')

print(f'XGBRegressor TRAINING...')
XGB_R = XGBRegressor(**XGB_R_parm)
#XGB_R.fit(train_x, train_y, eval_set = [(test_x, test_y)],eval_metric = "rmse", verbose = False)
joblib.dump(XGB_R, 'XGB_R.pkl')

print(f'RandomForestRegressor TRAINING...')
random_forest = RandomForestRegressor(**R_Forest_parm)
#random_forest.fit(train_x, train_y)
joblib.dump(random_forest, 'random_forest.pkl')

print(f'ExtraTreesRegressor TRAINING...')
extra_trees = ExtraTreesRegressor(**Extra_parm)
#extra_trees.fit(train_x, train_y)
joblib.dump(extra_trees, 'extra_trees.pkl')

print(f'GradientBoostingRegressor TRAINING...')
GradientBoosting = GradientBoostingRegressor(**GB_params)
#GradientBoosting.fit(train_x, train_y)
joblib.dump(GradientBoosting, 'GradientBoosting.pkl')

print(f'StackingRegressor TRAINING...')
estimators_1 = [('LGBMR', LGBM_R), ('CatBoost', catboost)] #
estimators_2 = [('random_forest', random_forest), ('extra_trees', extra_trees), ('GradientBoosting', GradientBoosting)]
        
model_0 = StackingRegressor(estimators_1 + estimators_2, final_estimator = RidgeCV())
#model_0.fit(train_x, train_y)
joblib.dump(model_0, f'model_0.pkl')

CatBoostRegressor TRAINING...
LGBMRegressor TRAINING...
XGBRegressor TRAINING...
RandomForestRegressor TRAINING...
ExtraTreesRegressor TRAINING...
GradientBoostingRegressor TRAINING...
StackingRegressor TRAINING...


['model_0.pkl']

In [5]:
catboost_model = joblib.load('catboost.pkl')
LGBM_R_model = joblib.load('LGBM_R.pkl')
XGB_R_model = joblib.load('XGB_R.pkl')
random_forest_model = joblib.load('random_forest.pkl')
extra_trees_model = joblib.load('extra_trees.pkl')
GradientBoosting_model = joblib.load('GradientBoosting.pkl')
model_0_model = joblib.load('model_0.pkl')

In [6]:
# Signal conversion parameters (like in working code)
SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    """Convert raw prediction to signal in range [0.0, 2.0]"""
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)

def predict(test: pl.DataFrame) -> float:
    try:
        # Convert to pandas
        test_df = test.to_pandas()
        
        # Preprocess - use 'inference' type to handle test data properly
        test_processed = preprocessing(test_df, 'test', add_lag_features=True)
        #print(test_processed)
        
        LGBM_R_y_pred = LGBM_R_model.predict(test_processed)
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        #print(float(signal))
        
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0  # Return neutral signal (midpoint) on error
    
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))